In [3]:
"""
Read data from the apache-logs.txt file
Load and display the data
"""

file_df = spark.read.format("text").load(path = "/workspaces/pyspark_udemy_codespace/data/apache-logs.txt")
file_df.show() # each line has become one row of a single column

+--------------------+
|               value|
+--------------------+
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
+--------------------+
only showing top 20 rows


In [4]:
"""
Develop an strategy to extract the following fields
    ip_address: It is the IP address of the site visitor.
    visit_timestamp: It is the date and time of the site visit. Parse and format the timestamp to YYYY-MM-DD HH:MI:SS Z
    visit_resource: Which resource from our website was accessed
    referring_url: It is the clean URL of the referring website.
"""

# develop a regex
log_reg = r'^(\S+) (\S+) (\S+) \[([\w:/]+\s[+\-]\d{4})\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+) "(\S+)" "([^"]*)'

# apply regex to parse record
from pyspark.sql.functions import regexp_extract

logs_df = (
    file_df.select(
        regexp_extract("value", log_reg, 1).alias("ip_address"),
        regexp_extract("value", log_reg, 4).alias("visit_timestamp"),
        regexp_extract("value", log_reg, 6).alias("visit_resource"),
        regexp_extract("value", log_reg, 10).alias("referring_url")
    )
)
# logs_df.show()

# refine results with other transformations
from pyspark.sql.functions import to_timestamp, col, substring_index
logs_refined_df = logs_df.withColumns({
    "visit_timestamp": to_timestamp("visit_timestamp", "dd/MMM/yyyy:HH:mm:ss Z"),
    "referring_url": substring_index(col("referring_url"), '/', 3)
})
logs_refined_df.show()

+------------+-------------------+--------------------+--------------------+
|  ip_address|    visit_timestamp|      visit_resource|       referring_url|
+------------+-------------------+--------------------+--------------------+
|83.149.9.216|2015-05-17 10:05:03|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:43|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:47|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:12|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:07|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:34|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:57|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:50|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:24|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:50|/presentations/lo...|http://semicomple...|

In [5]:
# using LLM to transform unstructred data
from pyspark.sql.functions import concat, col, expr

# prompt
prompt = """
    You will be provided with an Apache log file record. It is an unstructured text record. 
    Each record represents some information for our website visits, such as what is the IP address of the visitor, 
    What is the date and time of the visit, which resource was requested, and the URL of the referring website? 
    You are asked to parse the log file record and extract the following fields.
    ip_address: It is the IP address of the site visitor.
    visit_timestamp: It is the date and time of the site visit. Parse and format the timestamp to YYYY-MM-DD HH:MI:SS Z
    visit_resource: Which resource from our website was accessed?
    referring_url: It is the clean URL of the referring website. When the actual referring URL is not given, 
    you can extract the URL from the user agent. For cleaning the URL, you should take the values only up to the domain extension, 
    such as .com, .in, .uk, etc.
    Give only the final answer in the JSON format.
    Record:
"""

result_df = file_df.limit(10)\
                .withColumn("prompt", concat(prompt, col("value")))\
                .withColumn("json_extract", expr("""
                    ai_query(
                        endpoint=> 'databricks-llama-4-maverick',
                        request=> prompt,
                        responseFormat=> 'struct<extract: struct<
                        ip_address: string,
                        visit_timestamp: string,
                        visit_resource: string,
                        referring_url: string>>'
                    ))"""))

result_df.show()

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 46846)
Traceback (most recent call last):
  File "/usr/local/python/3.12.1/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/local/python/3.12.1/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/usr/local/python/3.12.1/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/local/python/3.12.1/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/accumulators.py", line 300, in handle
    poll(authenticate_and_accum_updates)
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/accumulators.py", line 272, in poll
    if self.rfile in r and func():
   

In [ ]:
from pyspark.sql.functions import from_json, selectExpr, to_timestamp

schema = "ip_address string, visit_timestamp timestamp, visit_resource string, referring_url string"

final_result_df = result_df.withColumn("json_extract", from_json(col("json_extract"), schema))\
                        .selectExpr("json_extract.*")\
                        .withColumn("visit_timestamp", to_timestamp(col("visit_timestamp", "yyyy-MM-dd HH:mm:ss Z")))
final_result_df.show()